## 7.7. Criando buffers de representatividade espacial para cada estação

In [21]:
#%% BUFFERS DE REPRESENTATIVIDADE -----------------------------------------------------
"""microscale: < 100 m
   mesoscale: 100 m < x < 500 m
   escala de bairro: 500 m < x < 4000 m
   escala urbana: 4000 m < x < 50000 m
""" 

def create_buffered_gdf(subset,
                        buffer_sizes,
                        target_crs="EPSG:4326",
                        create_buffer=True):
    """
    Cria colunas com tamanhos de buffer para um GeoDataFrame, agrupando por códigos EPSG,
    com a opção de criar geometrias de buffer.

    Parâmetros:
        subset (GeoDataFrame): GeoDataFrame de entrada com as colunas 'EPSG', 
        'REP_ESPACIAL_NAME' e 'geometry'.
        
        buffer_sizes (dict): Dicionário que mapeia os valores de `REP_ESPACIAL_NAME`
        para tamanhos de buffer (em metros).
        
        target_crs (str): CRS (sistema de referência espacial) para reprojetar o 
        GeoDataFrame final. Padrão é 'EPSG:4326'.
        
        create_buffer (bool): Verifica se as geometrias de buffer devem ser criadas. 
        Padrão é True.

    Retorna:
        GeoDataFrame: O GeoDataFrame de entrada com colunas adicionais:
            "REP_ESPACIAL": int.
                Tamanho do buffer de representatividade espacial
            "REP_ESPACIAL_NAME": str.
                Nome da classe de representatividade espacial 
            "REP_ESPACIAL_BUFFER": polígono. 
                Geometrias de buffer ao redor das estações no CRS de destino.
    """
    
    buffered_gdfs = []

    for epsg in subset['EPSG'].unique():
        # Filtra o subset e projeta para o respectivo EPSG
        gdf_epsg = subset[subset['EPSG'] == epsg].to_crs(epsg)

        # Remove a coluna REP_ESPACIAL existente
        gdf_epsg = gdf_epsg.drop(columns=['REP_ESPACIAL'])
        
        # Mapeia os tamanhos dos buffers com base no nome da representatividade
        gdf_epsg['REP_ESPACIAL'] = (
            gdf_epsg['REP_ESPACIAL_NAME']
            .map(buffer_sizes)
            .fillna(0)
        )
        
        # Cria as geometrias de buffer se create_buffer for True
        if create_buffer == True:
            gdf_epsg['REP_ESPACIAL_BUFFER'] = (
                gdf_epsg
                .geometry
                .buffer(gdf_epsg['REP_ESPACIAL'])
                .to_crs(target_crs)
            )

        # Reprojeta para o CRS de destino
        gdf_epsg = gdf_epsg.to_crs(target_crs)

        # Adiciona à lista
        buffered_gdfs.append(gdf_epsg)

    # Combina todos os GeoDataFrames com buffer
    buffered_subset = gpd.GeoDataFrame(
        pd.concat(buffered_gdfs, ignore_index=True),
        crs=target_crs
    )

    return buffered_subset


# Dicionário de tamanhos de buffer de acordo com REP_ESPACIAL_NAME
buffer_sizes = {
    'urbana': 50000,
    'bairro': 4000,
    'meso': 500,
    'micro': 100,
}

# Aplicando a função
# CO 
buffered_subset_co = create_buffered_gdf(subset_co,
                                         buffer_sizes,
                                         target_crs='EPSG:4326',
                                         create_buffer=False)
# NO2 
buffered_subset_no2 = create_buffered_gdf(subset_no2,
                                          buffer_sizes,
                                          target_crs='EPSG:4326',
                                          create_buffer=False)
# O3
buffered_subset_o3 = create_buffered_gdf(subset_o3,
                                         buffer_sizes,
                                         target_crs='EPSG:4326',
                                         create_buffer=False)
# PM 
buffered_subset_pm = create_buffered_gdf(subset_pm,
                                         buffer_sizes,
                                         target_crs='EPSG:4326',
                                         create_buffer=False)
# SO2 
buffered_subset_so2 = create_buffered_gdf(subset_so2,
                                          buffer_sizes,
                                          target_crs='EPSG:4326',
                                          create_buffer=False)


In [22]:
buffered_subset_co.head()

,UF,CIDADE,CD_MUN,ID_OEMA,ID_MMA,ID_MMA_COMPLETO,PROPRIETARIO,PROP_ENTIDADE,OPERADOR,OP_ENTIDADE,...,rep_micro_40k,rep_bairro_40k,rep_micro_50k,rep_bairro_50k,rep_micro_60k,rep_bairro_60k,rep_micro_any,rep_bairro_any,REP_ESPACIAL_NAME,REP_ESPACIAL
0,MT,CuiabÃÂ¡,5103403,Dom Aquino - BEA - CBA,NaN,NaN,NaN,NaN,NaN,NaN,...,False,True,False,True,False,False,False,True,bairro,4000
1,PR,Foz do IguaÃÂ§u,4108304,FIGUAÃÂU,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,False,False,False,False,True,bairro,4000
2,MT,VÃÂ¡rzea Grande,5108402,Água Limpa - CBM - VG,NaN,NaN,NaN,NaN,NaN,NaN,...,False,True,False,True,False,False,True,True,micro,100
3,MT,CuiabÃÂ¡,5103403,CPA - SEMA - CBA,NaN,NaN,SEMA,PÃºblica,UFMT e SEMA,PÃºblica,...,False,True,False,True,False,False,False,True,bairro,4000
4,MT,CuiabÃÂ¡,5103403,SEMA 2,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,True,False,False,False,True,bairro,4000
